# 项目概述与架构介绍

## 🎯 项目定位与特色

本项目是一个**智能优化框架**，其核心特色是通过偏置系统（Bias System）来引导和加速优化过程。与传统优化库不同，我们不追求支持最多的算法，而是专注于如何通过偏置机制让优化更智能、更高效。

### 🌟 核心创新点

1. **偏置驱动的优化** - 将先验知识、领域经验、约束条件转化为可计算的偏置
2. **统一的偏置接口** - 算法级偏置、领域级偏置、自适应偏置的统一框架
3. **算法-偏置-代理三重结合** - 优化算法、偏置策略、代理模型的深度集成
4. **灵活的扩展机制** - 易于接入新算法、新偏置、新问题类型

## 🏗️ 系统架构

```
SGABLACK 智能优化框架
│
├── 🎯 偏置系统 (Bias System) - [核心特色]
│   ├── bias/
│   │   ├── bias_base.py          # 偏置基类
│   │   ├── bias_v2.py           # 偏置系统v2.0
│   │   ├── bias_bayesian.py     # 贝叶斯偏置
│   │   ├── bias_library_algorithmic.py  # 算法偏置库
│   │   └── production_scheduling_bias.py # 领域偏置示例
│
├── 🔧 核心优化引擎 (Core)
│   ├── core/
│   │   ├── problems.py          # 标准问题定义
│   │   ├── solver.py            # 求解器基类
│   │   └── elite.py             # 精英保留机制
│
├── 🚀 算法实现 (Solvers)
│   ├── solvers/
│   │   ├── nsga2.py            # NSGA-II多目标优化
│   │   ├── moead.py            # MOEA/D分解算法
│   │   ├── monte_carlo.py      # 蒙特卡洛方法
│   │   ├── surrogate.py        # 代理模型辅助优化
│   │   ├── hybrid_bo.py        # 混合贝叶斯优化
│   │   └── vns.py              # 变邻域搜索
│
├── 📊 代理模型 (Surrogates)
│   ├── surrogate 模块集成在 solvers/
│   ├── 支持：KNN、随机森林、高斯过程等
│   └── 动态选择与组合策略
│
├── 🛠️ 工具集 (Utilities)
│   ├── utils/
│   │   ├── headless.py          # 可视化工具
│   │   ├── parallel_evaluator.py # 并行评估
│   │   ├── solver_extensions.py # 求解器扩展
│   │   ├── array_utils.py      # 数组工具
│   │   └── memory_manager.py   # 内存管理
│
├── 🧪 测试与示例 (Examples)
│   ├── examples/
│   │   ├── jupyter_tutorials/  # 详细教程
│   │   └── 各种算法示例
│
└── 📚 文档 (Docs)
    ├── docs/
    └── PROJECT_INTRODUCTION.md
```

In [ ]:
# 导入必要的库
import sys
import os
sys.path.append('../..')

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch
import seaborn as sns
from IPython.display import HTML, display, JSON
import json
import inspect
from pathlib import Path

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

print("✅ 环境配置完成")
print(f"📁 当前工作目录: {os.getcwd()}")
print(f"🐍 Python版本: {sys.version.split()[0]}")

## 1. 探索项目结构

In [ ]:
# 获取项目根目录
project_root = Path('../../').resolve()
print(f"📂 项目根目录: {project_root}")
print(f"📁 项目目录结构：")

# 递归显示目录结构
def show_tree(path, prefix="", max_depth=3, current_depth=0):
    if current_depth >= max_depth:
        return
    
    items = sorted([p for p in path.iterdir() if not p.name.startswith('.')])
    for i, item in enumerate(items):
        is_last = i == len(items) - 1
        
        if item.is_dir():
            print(f"{prefix}{'└── ' if is_last else '├── '}{item.name}/")
            new_prefix = prefix + "    " if is_last else prefix + "│   "
            show_tree(item, new_prefix, max_depth, current_depth + 1)
        else:
            # 只显示主要文件类型
            if item.suffix in ['.py', '.md', '.txt', '.json']:
                print(f"{prefix}{'└── ' if is_last else '├── '}{item.name}")

show_tree(project_root, max_depth=2)

# 统计各类文件
py_files = list(project_root.rglob('*.py'))
notebooks = list(project_root.rglob('*.ipynb'))
print(f"\n📊 项目统计：")
print(f"  • Python文件: {len(py_files)}个")
print(f"  • Jupyter笔记本: {len(notebooks)}个")

## 2. 核心模块导入与验证

In [ ]:
# 验证核心模块导入
print("🔍 验证核心模块导入...\n")

# 1. 偏置系统
try:
    from bias import *
    from bias.bias_base import BiasBase
    from bias.bias_v2 import AlgorithmicBias, DomainBias
    print("✅ 偏置系统模块导入成功")
    print(f"   • 偏置基类: {BiasBase.__name__}")
    print(f"   • 算法偏置: {AlgorithmicBias.__name__}")
    print(f"   • 领域偏置: {DomainBias.__name__}")
except Exception as e:
    print(f"❌ 偏置系统模块导入失败: {e}")

# 2. 核心模块
try:
    from core.problems import *
    from core.solver import SolverBase
    from core.elite import EliteArchive
    print("\n✅ 核心模块导入成功")
    print(f"   • 可用问题类型: {[cls.__name__ for cls in ProblemBase.__subclasses__()][:5]}...")
except Exception as e:
    print(f"\n❌ 核心模块导入失败: {e}")

# 3. 求解器
try:
    from solvers import *
    from solvers.nsga2 import NSGA2Optimizer
    from solvers.moead import MOEADOptimizer
    from solvers.monte_carlo import MonteCarloOptimizer
    print("\n✅ 求解器模块导入成功")
    print(f"   • NSGA-II: {NSGA2Optimizer.__name__}")
    print(f"   • MOEA/D: {MOEADOptimizer.__name__}")
    print(f"   • 蒙特卡洛: {MonteCarloOptimizer.__name__}")
except Exception as e:
    print(f"\n❌ 求解器模块导入失败: {e}")

# 4. 工具集
try:
    from utils.headless import Visualizer
    from utils.parallel_evaluator import ParallelEvaluator
    print("\n✅ 工具集模块导入成功")
    print(f"   • 可视化器: {Visualizer.__name__}")
    print(f"   • 并行评估器: {ParallelEvaluator.__name__}")
except Exception as e:
    print(f"\n❌ 工具集模块导入失败: {e}")

## 3. 可用算法与问题概览

In [ ]:
# 创建算法和问题的概览图
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# 左图：可用算法
ax1 = axes[0]
algorithms_info = {
    "NSGA-II": ["多目标", "精英保留", "快速非支配排序"],
    "MOEA/D": ["多目标", "分解策略", "权重向量"],
    "蒙特卡洛": ["全局搜索", "随机采样", "并行友好"],
    "贝叶斯优化": ["智能搜索", "代理模型", "探索利用平衡"],
    "代理优化": ["计算高效", "模型辅助", "自适应采样"],
    "变邻域搜索": ["局部搜索", "邻域结构", "跳出局部最优"],
    "混合算法": ["多策略", "自适应切换", "性能互补"]
}

# 绘制算法关系图
y_pos = 0.9
for alg_name, features in algorithms_info.items():
    # 算法框
    rect = FancyBboxPatch((0.05, y_pos - 0.05), 0.25, 0.08, 
                         boxstyle="round,pad=0.01", 
                         facecolor='lightblue', edgecolor='blue', linewidth=2)
    ax1.add_patch(rect)
    ax1.text(0.175, y_pos - 0.01, alg_name, ha='center', va='center', 
             fontsize=11, fontweight='bold')
    
    # 特征
    for i, feature in enumerate(features):
        ax1.text(0.35, y_pos - 0.01 - i*0.025, f"• {feature}", 
                fontsize=9, va='center')
    
    y_pos -= 0.12

ax1.set_xlim(0, 0.8)
ax1.set_ylim(0, 1)
ax1.set_title('🚀 可用优化算法', fontsize=14, fontweight='bold')
ax1.axis('off')

# 右图：标准问题库
ax2 = axes[1]
problems_info = {
    "单目标": {
        "Sphere": "全局最优，无局部",
        "Rastrigin": "多峰，欺骗性强",
        "Rosenbrock": "窄谷，病态",
        "Ackley": "多模态，指数项",
        "Griewank": "多峰，耦合项"
    },
    "多目标": {
        "ZDT系列": "连续，凸/凹",
        "DTLZ系列": "高维，可扩展",
        "WFG系列": "复杂，实用",
        "TSP": "组合优化",
        "调度问题": "实际应用"
    }
}

# 绘制问题分类
y_pos = 0.9
for category, problems in problems_info.items():
    # 分类标题
    ax2.text(0.1, y_pos, category, fontsize=12, fontweight='bold', 
             color='darkgreen')
    y_pos -= 0.06
    
    # 具体问题
    for prob_name, description in problems.items():
        ax2.text(0.15, y_pos, f"• {prob_name}: {description}", 
                fontsize=9)
        y_pos -= 0.04
    
    y_pos -= 0.03

ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)
ax2.set_title('📊 标准测试问题', fontsize=14, fontweight='bold')
ax2.axis('off')

plt.tight_layout()
plt.show()

# 显示更详细的信息
print("\n📋 详细信息：")
print(f"\n🔬 单目标测试函数：")
single_objective_problems = [cls.__name__ for cls in ProblemBase.__subclasses__() 
                             if hasattr(cls, 'n_obj') and cls().n_obj == 1]
for prob in single_objective_problems[:10]:
    print(f"  • {prob}")

print(f"\n🎯 多目标测试函数：")
multi_objective_problems = [cls.__name__ for cls in ProblemBase.__subclasses__() 
                            if hasattr(cls, 'n_obj') and cls().n_obj > 1]
for prob in multi_objective_problems[:10]:
    print(f"  • {prob}")

## 4. 偏置系统架构详解

In [ ]:
# 创建偏置系统架构可视化
fig, ax = plt.subplots(1, 1, figsize=(14, 10))

# 定义颜色
colors = {
    'base': '#E8F4FD',
    'algorithmic': '#FFE6CC',
    'domain': '#D5E8D4',
    'adaptive': '#F8CECC',
    'text': '#333333'
}

# 偏置系统主体
bias_main = FancyBboxPatch((0.3, 0.8), 0.4, 0.15,
                            boxstyle="round,pad=0.02",
                            facecolor=colors['base'],
                            edgecolor='blue', linewidth=3)
ax.add_patch(bias_main)
ax.text(0.5, 0.875, '偏置系统 (Bias System)', 
        ha='center', va='center', fontsize=16, fontweight='bold')

# 三大偏置类型
bias_types = [
    (0.1, 0.5, '算法级偏置', colors['algorithmic'], [
        '• 选择偏置',
        '• 变异偏置',
        '• 交叉偏置',
        '• 替换偏置'
    ]),
    (0.5, 0.5, '领域级偏置', colors['domain'], [
        '• 约束偏置',
        '• 偏好偏置',
        '• 启发式偏置',
        '• 经验偏置'
    ]),
    (0.7, 0.5, '自适应偏置', colors['adaptive'], [
        '• 动态调整',
        '• 学习机制',
        '• 反馈控制',
        '• 在线优化'
    ])
]

# 绘制偏置类型和连接
for x, y, title, color, features in bias_types:
    # 偏置类型框
    rect = FancyBboxPatch((x - 0.08, y - 0.08), 0.16, 0.16,
                         boxstyle="round,pad=0.01",
                         facecolor=color,
                         edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(x, y + 0.04, title, ha='center', va='center', 
            fontsize=11, fontweight='bold')
    
    # 连接线
    ax.plot([0.5, x], [0.8, y + 0.08], 'k-', linewidth=2)
    
    # 特征列表
    for i, feature in enumerate(features):
        ax.text(x, y - 0.15 - i*0.04, feature, 
                ha='center', va='center', fontsize=9)

# 偏置基类
bias_base = FancyBboxPatch((0.35, 0.15), 0.3, 0.1,
                           boxstyle="round,pad=0.01",
                           facecolor='#F0F0F0',
                           edgecolor='gray', linewidth=2,
                           linestyle='--')
ax.add_patch(bias_base)
ax.text(0.5, 0.2, 'BiasBase (基类)', 
        ha='center', va='center', fontsize=12, style='italic')

# 基类方法
base_methods = [
    'apply()     - 应用偏置',
    'initialize() - 初始化',
    'update()    - 更新参数',
    'configure() - 配置管理'
]

for i, method in enumerate(base_methods):
    ax.text(0.5, 0.1 - i*0.025, method, 
            ha='center', va='center', fontsize=9, 
            family='monospace')

# 连接基类到偏置系统
ax.plot([0.5, 0.5], [0.25, 0.5], 'k--', linewidth=1)

# 标题和说明
ax.set_title('偏置系统架构图', fontsize=18, fontweight='bold', pad=20)
ax.text(0.5, 0.02, '偏置系统是本框架的核心创新，将先验知识转化为可计算的优化策略', 
        ha='center', va='center', fontsize=11, style='italic')

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

plt.show()

# 探索偏置模块的具体实现
print("\n🔍 偏置模块结构：")
bias_dir = Path('../../bias')
if bias_dir.exists():
    for file in sorted(bias_dir.glob('*.py')):
        if not file.name.startswith('__'):
            print(f"  📄 {file.name}")
            # 简要读取文件内容获取类名
            try:
                with open(file, 'r', encoding='utf-8') as f:
                    content = f.read()
                    import re
                    classes = re.findall(r'^class\s+(\w+)\s*\(', content, re.MULTILINE)
                    for cls in classes[:3]:  # 只显示前3个类
                        print(f"    └─ {cls}")
            except:
                pass

## 5. 快速开始示例

In [ ]:
# 创建一个简单的优化示例
print("🚀 快速开始：优化Rastrigin函数\n")

# 1. 创建问题
problem = RastriginProblem(n_var=10, n_obj=1)
print(f"📊 问题: {problem.__class__.__name__}")
print(f"  • 维度: {problem.n_var}")
print(f"  • 变量范围: [{problem.xl[0]:.1f}, {problem.xu[0]:.1f}]")
print(f"  • 最优值: 0 (在原点)")

# 2. 配置和运行优化器（无偏置）
print("\n1️⃣ 标准优化（无偏置）:")
config_standard = {
    "pop_size": 50,
    "max_gen": 50,
    "bias_enabled": False
}

from solvers.nsga2 import NSGA2Optimizer
optimizer_std = NSGA2Optimizer(config=config_standard)

import time
start_time = time.time()
result_std = optimizer_std.optimize(problem)
time_std = time.time() - start_time

print(f"  ✅ 完成！耗时: {time_std:.2f}秒")
print(f"  🎯 最优值: {result_std.objectives[-1]:.4f}")
print(f"  📍 最优解: {result_std.decisions[-1][:3]}... (显示前3维)")

# 3. 配置和运行优化器（有偏置）
print("\n2️⃣ 偏置优化:")
config_biased = {
    "pop_size": 50,
    "max_gen": 50,
    "bias_enabled": True,
    "bias_configs": {
        "selection": {
            "bias_type": "elitist",
            "bias_strength": 0.7
        },
        "mutation": {
            "bias_direction": "improvement",
            "bias_strength": 0.5
        }
    }
}

optimizer_biased = NSGA2Optimizer(config=config_biased)

start_time = time.time()
result_biased = optimizer_biased.optimize(problem)
time_biased = time.time() - start_time

print(f"  ✅ 完成！耗时: {time_biased:.2f}秒")
print(f"  🎯 最优值: {result_biased.objectives[-1]:.4f}")
print(f"  📍 最优解: {result_biased.decisions[-1][:3]}... (显示前3维)")

# 4. 性能对比
improvement = (result_std.objectives[-1] - result_biased.objectives[-1]) / result_std.objectives[-1] * 100
print(f"\n📈 性能对比:")
print(f"  • 目标值改进: {improvement:.2f}%")
print(f"  • 时间效率: {time_std/time_biased:.2f}x")

# 5. 可视化对比
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 收敛曲线（模拟）
ax1 = axes[0]
generations = range(len(result_std.objectives))
ax1.plot(generations, result_std.objectives, 'b-', label='无偏置', linewidth=2)
ax1.plot(generations, result_biased.objectives, 'r-', label='有偏置', linewidth=2)
ax1.set_xlabel('评估次数')
ax1.set_ylabel('目标值')
ax1.set_title('收敛曲线对比')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

# 最终解分布
ax2 = axes[1]
values = [result_std.objectives[-1], result_biased.objectives[-1]]
labels = ['无偏置', '有偏置']
colors = ['blue', 'red']
bars = ax2.bar(labels, values, color=colors, alpha=0.7)
ax2.set_ylabel('最优目标值')
ax2.set_title('最终最优值对比')
ax2.grid(True, alpha=0.3, axis='y')

# 添加数值标签
for bar, val in zip(bars, values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height(), 
             f'{val:.2f}', ha='center', va='bottom')

# 时间对比
ax3 = axes[2]
times = [time_std, time_biased]
bars = ax3.bar(labels, times, color=colors, alpha=0.7)
ax3.set_ylabel('计算时间 (秒)')
ax3.set_title('计算时间对比')
ax3.grid(True, alpha=0.3, axis='y')

# 添加数值标签
for bar, t in zip(bars, times):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height(), 
             f'{t:.2f}s', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 6. 教程路线图

基于项目架构和特色，我们设计了以下学习路径：

### 📚 教程系列

| 章节 | 标题 | 核心内容 | 难度 |
|------|------|----------|------|
| 00 | **项目概述与架构介绍** | 整体架构、核心特色、快速开始 | ⭐ |
| 01 | **偏置系统核心概念详解** | 偏置原理、分类、设计模式 | ⭐⭐ |
| 02 | **蒙特卡洛算法与偏置结合** | 全局搜索、偏置引导、并行化 | ⭐⭐ |
| 03 | **NSGA-II算法详解与偏置应用** | 多目标优化、精英保留、偏置策略 | ⭐⭐⭐ |
| 04 | **贝叶斯优化与智能偏置** | 代理模型、采集函数、自适应偏置 | ⭐⭐⭐ |
| 05 | **算法嵌套与混合策略** | 算法组合、切换策略、性能优化 | ⭐⭐⭐ |
| 06 | **代理模型详解与应用** | KNN、随机森林、高斯过程、动态选择 | ⭐⭐⭐ |
| 07 | **高级优化组件** | 降维器、收敛器、历史解管理 | ⭐⭐⭐⭐ |
| 08 | **自定义偏置开发指南** | 继承基类、实现接口、最佳实践 | ⭐⭐⭐ |
| 09 | **约束处理与偏置** | 硬约束、软约束、奖励机制 | ⭐⭐⭐⭐ |
| 10 | **偏置-代理-算法三重结合** | 系统集成、高级应用、案例研究 | ⭐⭐⭐⭐⭐ |

### 🎯 学习建议

1. **循序渐进** - 按照章节顺序学习，每个章节都建立在前一章的基础上
2. **动手实践** - 每个教程都包含可运行的代码示例，请亲自运行和修改
3. **理解原理** - 不仅要会用，更要理解为什么这样设计
4. **创新应用** - 尝试将所学知识应用到自己的问题中

### 🔧 使用本框架

```python
# 基本使用模式
from your_problem import YourProblem
from solvers.your_algorithm import YourOptimizer
from bias.your_bias import YourBias

# 1. 定义问题
problem = YourProblem()

# 2. 配置偏置
bias = YourBias(config=bias_config)

# 3. 配置优化器
optimizer = YourOptimizer(config={
    "bias_enabled": True,
    "bias": bias
})

# 4. 运行优化
result = optimizer.optimize(problem)
```

In [ ]:
# 保存教程配置
tutorial_config = {
    "version": "1.0",
    "author": "SGABLACK Team",
    "date": "2024-01-01",
    "core_features": [
        "偏置驱动的优化",
        "统一的偏置接口",
        "算法-偏置-代理三重结合",
        "灵活的扩展机制"
    ],
    "tutorial_series": [
        "00_项目概述与架构介绍",
        "01_偏置系统核心概念详解",
        "02_蒙特卡洛算法与偏置结合",
        "03_NSGA-II算法详解与偏置应用",
        "04_贝叶斯优化与智能偏置",
        "05_算法嵌套与混合策略",
        "06_代理模型详解与应用",
        "07_高级优化组件",
        "08_自定义偏置开发指南",
        "09_约束处理与偏置",
        "10_偏置-代理-算法三重结合"
    ]
}

# 保存到文件
import json
with open('tutorial_config.json', 'w', encoding='utf-8') as f:
    json.dump(tutorial_config, f, ensure_ascii=False, indent=2)

print("\n💾 教程配置已保存")
print(f"📚 教程总数: {len(tutorial_config['tutorial_series'])}个")
print(f"🚀 准备开始学习！")

# 创建学习进度跟踪
progress = {tutorial: False for tutorial in tutorial_config['tutorial_series']}
progress['00_项目概述与架构介绍'] = True  # 标记当前章节为已完成

with open('learning_progress.json', 'w', encoding='utf-8') as f:
    json.dump(progress, f, ensure_ascii=False, indent=2)

print("\n✅ 学习进度跟踪已初始化")
print(f"📈 当前进度: {sum(progress.values())}/{len(progress)} 章节完成")

print("\n" + "="*60)
print("🎉 项目概述与架构介绍完成！")
print("\n下一步：")
print("  1. 学习「01_偏置系统核心概念详解」了解偏置原理")
print("  2. 尝试运行示例代码，熟悉项目结构")
print("  3. 思考自己的问题如何使用偏置系统")
print("="*60)

## 总结

### 🎯 核心要点回顾

1. **项目定位** - 智能优化框架，核心是偏置系统
2. **架构特色** - 偏置驱动、统一接口、三重结合、灵活扩展
3. **技术栈** - Python + NumPy + 科学计算可视化
4. **应用场景** - 单/多目标优化、约束优化、实际工程问题

### 🚀 准备好了吗？

现在您已经了解了项目的整体架构和核心特色。在接下来的教程中，我们将深入探讨：

- 偏置系统的工作原理
- 如何为不同问题设计偏置
- 算法与偏置的最佳实践
- 高级应用和扩展技巧

### 📞 需要帮助？

- 查看项目文档：`docs/` 目录
- 运行示例代码：`examples/` 目录
- 提交问题：GitHub Issues

让我们开始这段精彩的优化之旅吧！ 🎊